In [3]:
from pathlib import Path
import pandas as pd
from IPython.display import display

project_dir = Path("~/Developer/project/monster-hack").expanduser()
train_dir = project_dir / "Dataset" / "Dataset" / "Training"

people_path = train_dir / "la_mmh_training_person_dim.csv"
lms_path = train_dir / "la_mmh_training_lms_item_activity_by_session_fact.csv"
video_path = train_dir / "la_mmh_training_video_activity_by_view_fact.csv"

for path in [people_path, lms_path, video_path]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing file: {path}")

In [5]:
people = pd.read_csv(
    people_path,
    dtype={"P_ID": "string"},
)

for column in ["registration_date", "deregistration_date"]:
    people[column] = pd.to_datetime(
        people[column],
        errors="raise",
    )

print(f"Loaded {len(people):,} student rows")
display(people.head())
people.info()

display(
    people.isna().sum().rename("missing_values").to_frame()
)

Loaded 14,951 student rows


,P_ID,registration_date,deregistration_date
0,P00001187,2023-09-01,NaT
1,P00006491,2023-09-21,NaT
2,P00007122,2023-09-05,NaT
3,P00001901,2023-09-06,NaT
4,P00009899,2023-09-10,NaT


<class 'pandas.DataFrame'>
RangeIndex: 14951 entries, 0 to 14950
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   P_ID                 14951 non-null  string        
 1   registration_date    14951 non-null  datetime64[us]
 2   deregistration_date  196 non-null    datetime64[us]
dtypes: datetime64[us](2), string(1)
memory usage: 350.5 KB


,missing_values
P_ID,0
registration_date,0
deregistration_date,14755


In [7]:
deregistered = people.loc[
    people["deregistration_date"].notna(),
    ["P_ID", "deregistration_date"],
].copy()

# Remove identical repeated records before matching activity.
deregistered = deregistered.drop_duplicates()

# Different deregistration dates for one ID require investigation.
if deregistered["P_ID"].duplicated().any():
    raise ValueError(
        "Some students have multiple deregistration dates. "
        "Inspect these records before continuing."
    )

print(f"{len(deregistered):,} students to investigate")
display(deregistered.head())

196 students to investigate


,P_ID,deregistration_date
15,P00021856,2024-02-28
96,P00018806,2024-02-28
337,P00001260,2024-01-08
391,P00015757,2024-04-13
594,P00001716,2024-01-18


In [9]:
def find_post_deregistration_activity(
    csv_path,
    source_name,
    deregistered_students,
    chunksize=200_000,
):
    student_ids = set(deregistered_students["P_ID"])
    summaries = []
    rows_scanned = 0

    chunks = pd.read_csv(
        csv_path,
        usecols=["P_ID", "DATE"],
        dtype={"P_ID": "string", "DATE": "string"},
        chunksize=chunksize,
    )

    for chunk in chunks:
        rows_scanned += len(chunk)

        # Only investigate students with a deregistration date.
        relevant = chunk.loc[
            chunk["P_ID"].isin(student_ids)
        ].copy()

        if relevant.empty:
            continue

        relevant["activity_date"] = pd.to_datetime(
            relevant["DATE"],
            errors="raise",
        ).dt.normalize()

        if relevant["activity_date"].isna().any():
            raise ValueError(
                f"{source_name}: some relevant activity dates are missing."
            )

        matched = relevant.merge(
            deregistered_students,
            on="P_ID",
            how="inner",
            validate="many_to_one",
        )

        # Compare calendar dates:
        # activity on the deregistration date itself is not flagged.
        after = matched.loc[
            matched["activity_date"]
            > matched["deregistration_date"].dt.normalize()
        ]

        if after.empty:
            continue

        summary = (
            after.groupby("P_ID", as_index=False)
            .agg(
                deregistration_date=("deregistration_date", "first"),
                first_activity_after=("activity_date", "min"),
                last_activity_after=("activity_date", "max"),
                activity_rows_after=("activity_date", "size"),
            )
        )

        summaries.append(summary)

    columns = [
        "P_ID",
        "deregistration_date",
        "first_activity_after",
        "last_activity_after",
        "activity_rows_after",
        "source",
    ]

    if not summaries:
        print(
            f"{source_name}: scanned {rows_scanned:,} rows; "
            "no matches found."
        )
        return pd.DataFrame(columns=columns)

    # A student's activity can occur in several chunks.
    # Combine those partial summaries into one result.
    result = (
        pd.concat(summaries, ignore_index=True)
        .groupby("P_ID", as_index=False)
        .agg(
            deregistration_date=("deregistration_date", "first"),
            first_activity_after=("first_activity_after", "min"),
            last_activity_after=("last_activity_after", "max"),
            activity_rows_after=("activity_rows_after", "sum"),
        )
    )

    result["source"] = source_name

    print(
        f"{source_name}: scanned {rows_scanned:,} rows; "
        f"found {len(result):,} matching students."
    )

    return result[columns]

In [11]:
lms_evidence = find_post_deregistration_activity(
    lms_path,
    "LMS",
    deregistered,
)

video_evidence = find_post_deregistration_activity(
    video_path,
    "Video",
    deregistered,
)

nonempty_results = [
    result
    for result in [lms_evidence, video_evidence]
    if not result.empty
]

if nonempty_results:
    undead_evidence = (
        pd.concat(nonempty_results, ignore_index=True)
        .sort_values(["P_ID", "source"])
        .reset_index(drop=True)
    )
else:
    undead_evidence = lms_evidence.copy()

print(
    "\nTotal unique Undead Student candidates:",
    undead_evidence["P_ID"].nunique(),
)

display(undead_evidence.head(20))

LMS: scanned 7,857,424 rows; found 18 matching students.
Video: scanned 1,145,358 rows; found 79 matching students.

Total unique Undead Student candidates: 92


,P_ID,deregistration_date,first_activity_after,last_activity_after,activity_rows_after,source
0,P00000174,2023-12-14,2024-01-20,2024-01-24,10,Video
1,P00000673,2024-03-19,2024-04-14,2024-04-19,18,Video
2,P00000814,2024-02-25,2024-04-09,2024-04-16,18,Video
3,P00001095,2024-02-18,2024-02-20,2024-02-24,23,LMS
4,P00001260,2024-01-08,2024-02-06,2024-03-02,11,Video
5,P00001322,2023-10-29,2023-11-03,2023-11-03,1,Video
6,P00001620,2023-10-17,2023-10-31,2023-10-31,1,Video
7,P00001716,2024-01-18,2024-01-22,2024-02-07,2,Video
8,P00001868,2024-04-10,2024-04-11,2024-04-17,22,Video
9,P00002095,2024-01-31,2024-03-15,2024-03-15,1,Video


In [13]:
undead_submission = (
    undead_evidence[["P_ID"]]
    .drop_duplicates()
    .sort_values("P_ID")
    .reset_index(drop=True)
)

undead_submission.insert(
    0,
    "Monster Type",
    "Undead Student",
)

print(f"{len(undead_submission):,} unique candidates")
display(undead_submission)

92 unique candidates


,Monster Type,P_ID
0,Undead Student,P00000174
1,Undead Student,P00000673
2,Undead Student,P00000814
3,Undead Student,P00001095
4,Undead Student,P00001260
...,...,...
87,Undead Student,P00021224
88,Undead Student,P00021399
89,Undead Student,P00021856
90,Undead Student,P00022023


In [14]:
if not undead_submission.empty:
    example_id = undead_submission.loc[0, "P_ID"]

    print("Example candidate:", example_id)

    display(
        undead_evidence.loc[
            undead_evidence["P_ID"] == example_id
        ]
    )
else:
    print("No candidates found. Review the dates and source data.")

Example candidate: P00000174


,P_ID,deregistration_date,first_activity_after,last_activity_after,activity_rows_after,source
0,P00000174,2023-12-14,2024-01-20,2024-01-24,10,Video


## Stages 2 and 3: shared detector suite

**Undead Student**: Activity DATE strictly after deregistration calendar date; UK day-first person dates.

Person dates are parsed day-first. Activity on the deregistration date itself is not flagged. Activity-only IDs have no deregistration evidence.

Predictions come from every person/activity ID in this split, without reading HUNT labels. Training labels are used only for evaluation. UTC timestamps are used. The shared engine computes all seven detectors in one run. These saved results are from an executed run; set `REBUILD = True` to regenerate both splits.

Official TEST 1 accuracy is unknown. Workbook and slides are local outputs, not submitted files.

In [1]:
from pathlib import Path
import subprocess
import sys
import json

project_dir = Path.cwd()
if not (project_dir / "remaining_detectors.py").is_file():
    raise FileNotFoundError("Run from the monster-hack project directory")
REBUILD = False  # True reruns all seven detectors for Training and Test.
output_dir = project_dir / "outputs" / "undead"
if REBUILD or not all((output_dir / f"{split}_summary.json").exists() for split in ["training", "test"]):
    for split in ["Training", "Test"]:
        subprocess.run([sys.executable, "remaining_detectors.py", "--split", split], check=True)
for split in ["training", "test"]:
    summary = json.loads((output_dir / f"{split}_summary.json").read_text())
    print(split, "population:", summary["people"], "candidates:", summary["predicted"])
    if split == "training":
        score = summary["training_evaluation"]
        print("Training matched / extra / missed:", score["true_positives"], score["false_positives"], score["false_negatives"])
        print("Label IDs absent from inputs:", len(score["labels_absent_from_inputs"]))


training population: 14981 candidates: 92
Training matched / extra / missed: 92 0 0
Label IDs absent from inputs: 0
test population: 1879 candidates: 11


In [2]:
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

evidence = pd.read_csv(output_dir / "test_evidence.csv")
display(evidence.loc[evidence["is_monster"]].reset_index(drop=True))
print("Full IDs:", output_dir / "test_predictions.csv")
print("Workbook:", output_dir / "TEST1_Submission.xlsx")


         P_ID deregistration_date  ...  last_activity_after is_monster
0   P00000458          2024-03-05  ...           2024-03-12       True
1   P00003570          2024-02-09  ...           2024-03-19       True
2   P00003752          2024-02-01  ...           2024-02-09       True
3   P00004671          2023-12-28  ...           2024-02-08       True
4   P00005478          2024-03-25  ...           2024-04-23       True
5   P00006291          2024-02-29  ...           2024-03-19       True
6   P00010605          2023-11-09  ...           2023-11-12       True
7   P00010633          2024-02-10  ...           2024-02-14       True
8   P00010747          2024-02-03  ...           2024-03-13       True
9   P00015545          2024-01-28  ...           2024-03-19       True
10  P00018076          2024-04-06  ...           2024-04-24       True

[11 rows x 6 columns]
Full IDs: /Users/droitxenon/Developer/project/monster-hack/outputs/undead/test_predictions.csv
Workbook: /Users/droitxenon/De